## pypacity

### Examples of calculation

author: Mario Mañana 
last update: 4/8/2024

In [131]:
import sys
print( sys.version)
print(sys.path)

3.9.15 (main, Nov 24 2022, 14:39:17) [MSC v.1916 64 bit (AMD64)]
['e:\\mario\\python\\pypacity', 'c:\\Python37', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\python39.zip', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\DLLs', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\lib', 'c:\\Users\\manan\\anaconda3\\envs\\py39', '', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\lib\\site-packages', 'e:\\mario\\python\\jupyter-dash', 'e:\\mario\\trabajos2\\iberdrola_dtr\\datos\\eccodes-python', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\lib\\site-packages\\win32', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\lib\\site-packages\\win32\\lib', 'c:\\Users\\manan\\anaconda3\\envs\\py39\\lib\\site-packages\\Pythonwin']


In [132]:
from cable import cable
from case import case
from ieee738 import ieee738
from cigre601 import cigre601
from cigre207 import cigre207
from pvsystems import pvsystems
import matplotlib.pyplot as plt 

import numpy as np
import pandas as pd

from datetime import datetime, timedelta

# Needed only during the development phase.
from importlib import reload
reload(  cable)
reload(  case)
reload( ieee738)
reload( cigre601)
reload( cigre207)
reload( pvsystems)

<module 'pvsystems.pvsystems' from 'e:\\mario\\python\\pypacity\\pvsystems\\pvsystems.py'>

In [133]:
# Cargar el archivo Excel
excel_path = "DATOS.xlsx"  # Reemplaza con la ruta correcta de tu archivo Excel
data1 = pd.read_excel(excel_path, sheet_name= 'Hoja 1')  # Open meteo 
data2 = pd.read_excel(excel_path, sheet_name='Hoja 2')  # World Weather
data3 = pd.read_excel(excel_path, sheet_name='Hoja 3')  # Meteo Stat
data4 = pd.read_excel(excel_path, sheet_name='Hoja 4')  # Weather Bit 
data5 = pd.read_excel(excel_path, sheet_name='Medias')  # Medias

In [134]:
print( data5)

         fecha  hora       temp  velviento   dirviento  radiacion solar  \
0   2024-06-01     0  14.981667   5.115196  137.320831              NaN   
1   2024-06-01     1  14.898333   4.608307  235.101597              NaN   
2   2024-06-01     2  14.731667   5.190413  316.141246              NaN   
3   2024-06-01     3  14.165000   5.464558  251.512589              NaN   
4   2024-06-01     4  13.848333   5.541127  260.064824              NaN   
..         ...   ...        ...        ...         ...              ...   
451 2024-06-19    19  17.381667  11.288984  280.303711              NaN   
452 2024-06-19    20  17.215000  10.913334  274.333333              NaN   
453 2024-06-19    21  16.915000   9.993517  267.823110              NaN   
454 2024-06-19    22  17.031666  11.141369  265.124329              NaN   
455 2024-06-19    23  16.931666  10.993875  256.613419              NaN   

     hora solar  
0          0.00  
1          1.00  
2          2.00  
3          3.00  
4        

In [135]:
data5.columns

Index(['fecha', 'hora', 'temp', 'velviento', 'dirviento', 'radiacion solar',
       'hora solar'],
      dtype='object')

In [136]:
# número de registros en data5
print("Numero de datos sin filtrar: " + str(len(data5)))  # Print number of rows in data5

# filtrar filas con ciertos campos na
campos_filtro =['fecha', 'hora', 'temp', 'velviento', 'dirviento', 'hora solar']
# Eliminar filas con valores nulos en las columnas campos_filtro
data5_cleaned = data5.dropna(subset=campos_filtro)
print("Numero de datos filtrados: " + str(len(data5_cleaned)))  # Print number of rows in data5_cleaned

Numero de datos sin filtrar: 456
Numero de datos filtrados: 456


In [137]:
#for index, row in data2.iterrows():
#    dia_str = str(row['fecha'])  # Convert Timestamp to string
#    dia_int=dia_str[8:10]
#    #print(dia_int)  # Slice the string from the 3rd character onward

variable_names = [
    'Steady-state temperature',
    'Steady-state current',
    'Solar heating',
    'Radiation cooling',
    'Convection cooling'
]

#results_ieee738 = []
#results_cigreTB207 = []
#results_cigreTB601 = []

In [138]:
# Ejemplo de manejo de fecha y hora

#print( data5)

indice = 10

f1 = data5_cleaned.iloc[ indice]['fecha']
print( type( f1))

d1 = data5_cleaned.iloc[ indice]['hora']
print( type( d1))

# Crear un objeto timedelta con el número de horas
time_delta = timedelta(hours= int(d1))

# Sumar el timedelta a la fecha original para obtener un objeto único que integra fecha y hora
new_timestamp = f1 + time_delta
print( new_timestamp)

print("dia: " + str(new_timestamp.day) + "; mes: " + str( new_timestamp.month))

<class 'pandas._libs.tslibs.timestamps.Timestamp'>
<class 'numpy.int64'>
2024-06-01 10:00:00
dia: 1; mes: 6


In [139]:
NSELECT = 2 
Cable1 = cable.Cable()
c_db, error = Cable1.load_cable_db()
Cable1.set_cable( NSELECT, conductor = 'DRAKE')
Cable1.EMISS = 0.8
Cable1.ABSORP = 0.8


Celda combinada con todos los métodos

In [140]:
results_ieee738 = []
results_cigreTB207 = []
results_cigreTB601 = []
variable_names = ['TCDRPRELOAD', 'TR', 'QS', 'QR', 'QC']

for index, row in data5_cleaned.iloc[:10].iterrows(): #data5.iterrows():
    
    
    f1 = row['fecha']   # fecha
    #print( type( f1))

    h1 = row['hora'] # hora
    #print( type( d1))

    # Crear un objeto timedelta con el número de horas
    time_delta = timedelta(hours= int(h1))

    # Sumar el timedelta a la fecha original para obtener un objeto único que integra fecha y hora
    new_f1 = f1 + time_delta # new_f1 es la fecha con hora incluida en un único objeto
    mes = new_f1.month
    dia = new_f1.day
    print("                        ")
    print("                        ")
    print("---------------------------------------------------------------")
    print("Fecha: " + str(new_f1) + "; dia: " + str(dia) + "; mes: " + str(mes))
        
    
    #dia_str = str(row['fecha'])  # Convert Timestamp to string
    #dia_int=int(dia_str[8:10])
    #print(dia_int)
    PV1 = pvsystems.PVSystems()
    
    Case1 = case.Case()
    Case1.demo(NSELECT)
    # Ambient conditions
    Case1.TAMB = row['temp']
    print("Tamb: " + str(Case1.TAMB))
    Case1.CDR_LAT_DEG = 43.472501
    Case1.ALBEDO = 0.2
    Case1.beta = 0
    Case1.CDR_ELEV = 22.0
    Case1.TCDRPRELOAD = 100.0
    #Case1.TCDRMAX = 150
    #Case1.TCDR = 100.0
    Case1.VWIND = row['velviento']
    print("Vel viento: " + str(Case1.VWIND))
    Case1.DWIND_DEG = row['dirviento']
    print("Dir Viento: " + str(Case1.DWIND_DEG))
    Case1.Z1_DEG = 90.0
    print("Dir Eje línea: " + str(Case1.Z1_DEG))
    Case1.Ns = 1.0
    Case1.SUN_TIME = row['hora solar']
    
    Case1.NDAY = PV1.DayOfYear( dia, mes) 
    #print("NDAY: " + str(Case1.NDAY))
    print(f"Processing row {index}: NDAY = {Case1.NDAY}" + "; Hour = " + str(new_f1.hour))
    

    # IEEE 738
    X1 = ieee738.IEEE738()
    X1.Debug = 0
    X1.set_cable(Cable1)
    X1.set_case(Case1)
    X1.Case1.SORM = 1
    X1.ieee_738_2013()
    X1.outputs()
    #Steady-state temperature,Steady-state current,Solar heating,Radiation cooling,Convection cooling
    resultado = [
        X1.Case1.TCDRPRELOAD,
        X1.Case1.TR,
        X1.Case1.QS,
        X1.Case1.QR,
        X1.Case1.QC
    ]
    results_ieee738.append(resultado)

    # Print results
    #if NSELECT == 3:
    #    plt.plot(X1.Case1.TIME, X1.Case1.ATCDR)
    #if NSELECT == 4:
    #    print('The conductor temperature goes from {ti} to {tf} in {tt} minutes with a current of {i}'.format(ti=X1.Case1.TCDRPRELOAD, tf=X1.Case1.TCDRMAX, tt=X1.Case1.TT, i=X1.Case1.XISTEP))

    # CIGRE TB601
    X2 = cigre601.CIGRE601()
    X2.Debug = 0
    X2.set_cable( Cable1)
    X2.set_case( Case1)
    X2.Case1.SOLAR = 1 # compute solar radiation
    X2.cigre601()
    X2.output()

    #Steady-state temperature,Steady-state current,Solar heating,Radiation cooling,Convection cooling
    resultado = [
        X2.Case1.TCDRPRELOAD,
        X2.Case1.TR,
        X2.Case1.QS,
        X2.Case1.QR,
        X2.Case1.QC
    ]
    results_cigreTB601.append(resultado)


                        
                        
---------------------------------------------------------------
Fecha: 2024-06-01 00:00:00; dia: 1; mes: 6
Tamb: 14.9816665013631
Vel viento: 5.11519624392192
Dir Viento: 137.32083066304526
Dir Eje línea: 90.0
Processing row 0: NDAY = 152; Hour = 0
 
****************************************************************
*******************************************************************
IEEE 738
*******************************************************************
The angle between wind and conductor is =  47.32083066304526  DEG
INPUT -> Steady-state temperature:  100.0  ºC
OUTPUT -> Steady-state current:  2009.407  A
Solar heating:   0.0  W/m
Radiation cooling:  49.969  W/m
Convection cooling:  329.233  W/m
 
 
*******************************************************************
*******************************************************************
CIGRE TB601 
*******************************************************************
The angle between w

In [ ]:


# Verificar el número de resultados
print(f"Número de resultados recopilados: {len(results_ieee738)}")

# Crear un DataFrame a partir de los resultados
df_results_ieee738 = pd.DataFrame(results_ieee738, columns=variable_names)
print(f"Número de filas en data1: {len(data1)}")

# Guardar el DataFrame en un archivo Excel
output_file = 'resultados_ieee738.xlsx'
df_results_ieee738.to_excel(output_file, index=False)

print(f"Resultados guardados en {output_file}")

### CIGRE TB207

In [ ]:
X3 = cigre207.CIGRE207()
X3.Debug = 0
X3.set_cable( Cable1)
X3.set_case( Case1)
X3.cigre207()
X3.output()

## Steady state conductor temperature

In [ ]:
X1.Case1.NSELECT = 1
X1.Case1.XIPRELOAD = 1500.0
X1.ieee_738_2013()
X1.outputs()

In [ ]:
X2.Case1.NSELECT = 1
X2.Debug = 0
X2.cigre601()
X2.output()

In [ ]:
print(X2.Case1.XIPRELOAD)
print(X2.Case1.TCDRPRELOAD)

## Transient

In [ ]:
PV1 = pvsystems.PVSystems()
Case1 = case.Case()
NSELECT = 3
Case1.demo( NSELECT)
# Ambient conditions
Case1.TAMB = 40.0
Case1.CDR_LAT_DEG = 30
Case1.ALBEDO = 0.1
Case1.beta = 0
Case1.CDR_ELEV = 0
Case1.TCDRPRELOAD = 100
#Case1.TCDRMAX = 150
#Case1.TCDR = 100.0
Case1.WINDANG_DEG = 60
Case1.Z1_DEG = 90
Case1.Ns = 1.0
Case1.SUN_TIME = 11
Case1.NDAY = PV1.DayOfYear( 10, 6) # 10th June

In [ ]:
# IEEE 738
#X1 = ieee738.IEEE738()
X1.set_case( Case1)
X1.Debug = 0
#X1.set_cable( Cable1)
#X1.set_case( Case1)
X1.Case1.ATCDR = []
X1.Case1.TIME = []
X1.Case1.NSELECT = 3
X1.Case1.XIPRELOAD = 1000
X1.Case1.XISTEP = 100

X1.Case1.TT = 6000
X1.Case1.SORM = 1
X1.Case1.DELTIME = 60

X1.ieee_738_2013()
X1.outputs()



plt.plot( X1.Case1.TIME, X1.Case1.ATCDR)
ATCDRieee = X1.Case1.ATCDR
Time = X1.Case1.TIME

#

In [ ]:
X2.Case1.NSELECT = 3
X2.Case1.XIPRELOAD = 1000
X2.Case1.XISTEP = 100
X2.Case1.TT = 100
X2.Case1.SORM = 1
X2.Case1.DELTIME = 60
X2.Case1.TIME = []
X2.Case1.ATCDR = []
X2.Debug = 0
X2.cigre601()
X2.outputs()

plt.plot( X2.Case1.TIME, X2.Case1.ATCDR)
ATCDRcigre = X2.Case1.ATCDR 


In [ ]:
#plt.plot( Time, ATCDRieee, Time, ATCDRcigre)

len(ATCDRcigre)